# Sources
## About feature extraction (Bag of words, TD-IDF): 
https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction

# Predict tags on StackOverflow with linear models

In this assignment you will learn how to predict tags for posts from [StackOverflow](https://stackoverflow.com). To solve this task you will use multilabel classification approach.

### Libraries

In this task you will need the following libraries:
- [Numpy](http://www.numpy.org) — a package for scientific computing.
- [Pandas](https://pandas.pydata.org) — a library providing high-performance, easy-to-use data structures and data analysis tools for the Python
- [scikit-learn](http://scikit-learn.org/stable/index.html) — a tool for data mining and data analysis.
- [NLTK](http://www.nltk.org) — a platform to work with natural language.

### Data

The data required for this assignment is expected to be located in the folder `data`.

In [ ]:

dir_path = !pwd 
print(dir_path)

### Text preprocessing

For this and most of the following assignments you will need to use a list of stop words. It can be downloaded from *nltk*:

In [ ]:
import nltk
# nltk.download('stopwords', download_dir=dir_path[0])
nltk.download('stopwords') # install NLTK data to home user directory
from nltk.corpus import stopwords

In this task you will deal with a dataset of post titles from StackOverflow. You are provided a split to 3 sets: *train*, *validation* and *test*. All corpora (except for *test*) contain titles of the posts and corresponding tags (100 tags are available). Upload the corpora using *pandas* and look at the data:

In [ ]:
from ast import literal_eval
import pandas as pd
import numpy as np

In [ ]:
def read_data(filename):
    data = pd.read_csv(filename, sep='\t')
    data['tags'] = data['tags'].apply(literal_eval)
    return data

In [ ]:
train = read_data('data/train.tsv')
validation = read_data('data/validation.tsv')
test = pd.read_csv('data/test.tsv', sep='\t')

In [ ]:
train.head()

As you can see, *title* column contains titles of the posts and *tags* column contains the tags. It could be noticed that a number of tags for a post is not fixed and could be as many as necessary.

For a more comfortable usage, initialize *X_train*, *X_val*, *X_test*, *y_train*, *y_val*.

In [ ]:
X_train, y_train = train['title'].values, train['tags'].values
X_val, y_val = validation['title'].values, validation['tags'].values
X_test = test['title'].values

One of the most known difficulties when working with natural data is that it's unstructured. For example, if you use it "as is" and extract tokens just by splitting the titles by whitespaces, you will see that there are many "weird" tokens like *3.5?*, *"Flip*, etc. To prevent the problems, it's usually useful to prepare the data somehow. In this task you'll write a function, which will be also used in the other assignments. 

**Task 1 (TextPrepare).** Implement the function *text_prepare* following the instructions. After that, run the function *test_text_prepare* to test it on tiny cases.

In [ ]:
import re

In [ ]:
REPLACE_BY_SPACE_RE = re.compile('[/(){}\[\]\|@,;]')
BAD_SYMBOLS_RE = re.compile('[^0-9a-z #+_]')
STOPWORDS = set(stopwords.words('english'))
STOPWORDS_RE = re.compile(r"\b(" + "|".join(STOPWORDS) + ")\\W")

def text_prepare(text):
    """
        text: a string
        
        return: modified initial string
    """
    
    #text = # lowercase text
    text = text.lower()
    #text = # replace REPLACE_BY_SPACE_RE symbols by space in text
    text = REPLACE_BY_SPACE_RE.sub(" ", text)
    #text = # delete symbols which are in BAD_SYMBOLS_RE from text
    text = BAD_SYMBOLS_RE.sub("", text)
    # remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    #text = # delete stopwords from text
    text = STOPWORDS_RE.sub("", text)

    return text

In [ ]:
msg="SQL Server - any equivalent of Excel's CHOOSE function?"
print(text_prepare(msg))


In [ ]:
def test_text_prepare():
    examples = ["SQL Server - any equivalent of Excel's CHOOSE function?",
                "How to free c++ memory vector<int> * arr?"]
    answers = ["sql server equivalent excels choose function", 
               "free c++ memory vectorint arr"]
    for ex, ans in zip(examples, answers):
        if text_prepare(ex) != ans:
            return "Wrong answer for the case: '%s'" % ex
    return 'Basic tests are passed.'

In [ ]:
print(test_text_prepare())

Run your implementation for questions from file *text_prepare_tests.tsv* .

In [ ]:
prepared_questions = []
for line in open('data/text_prepare_tests.tsv', encoding='utf-8'):
    line = text_prepare(line.strip())
    prepared_questions.append(line)
text_prepare_results = '\n'.join(prepared_questions)

# save prepared questions on file
f = open("text_prepare_results.txt", "a")
f.write(text_prepare_results)
f.close()

Now we can preprocess the titles using function *text_prepare* and  making sure that the headers don't have bad symbols:

In [ ]:
X_train = [text_prepare(x) for x in X_train]
X_val = [text_prepare(x) for x in X_val]
X_test = [text_prepare(x) for x in X_test]

In [ ]:
X_train[:3]

For each tag and for each word calculate how many times they occur in the train corpus. 

**Task 2 (WordsTagsCount).** Find 3 most popular tags and 3 most popular words in the train data.

In [ ]:
# Dictionary of all tags from train corpus with their counts.
tags_counts = {}
# Dictionary of all words from train corpus with their counts.
words_counts = {}

# linearize list of tags
tagsList = [item for sublist in y_train for item in sublist]
# create dictionary of tags with their counts
tags_counts = dict((x,tagsList.count(x)) for x in set(tagsList))
# Create a list of tuples sorted by index 1 i.e. value field 
tagsTuplesSorted = sorted(tags_counts.items() , reverse=True, key=lambda x: x[1])
# Iterate over the sorted sequence
#for elem in tagsTuplesSorted :
#    print(elem[0] , " ::" , elem[1] ) 

print("Number of tags: ", len(tags_counts))

# create a list of words for each title in X_train
wordsList = []
for x in X_train:
    wordsList.append(re.sub(r'\s+', " ", x).split())
# linearize list of words
wordsList = [item for sublist in wordsList for item in sublist]
# create dictionary of words with their counts
words_counts = dict((x,wordsList.count(x)) for x in set(wordsList))
# Create a list of tuples sorted by index 1 i.e. value field 
wordsTuplesSorted = sorted(words_counts.items() , reverse=True, key=lambda x: x[1])

print("Number of words: ", len(words_counts))



We are assuming that *tags_counts* and *words_counts* are dictionaries like `{'some_word_or_tag': frequency}`. After applying the sorting procedure, results will be look like this: `[('most_popular_word_or_tag', frequency), ('less_popular_word_or_tag', frequency), ...]`. The grader gets the results in the following format (two comma-separated strings with line break):

    tag1,tag2,tag3
    word1,word2,word3


In [ ]:
most_common_tags = sorted(tags_counts.items(), key=lambda x: x[1], reverse=True)[:3]
most_common_words = sorted(words_counts.items(), key=lambda x: x[1], reverse=True)[:3]

print("Most common tags:")
print(most_common_tags)
print("Most common words")
print(most_common_words)
print("Task 2 output:")
print(','.join(tag for tag, _ in most_common_tags))
print(','.join(tag for tag, _ in most_common_words))


### Transforming text to a vector

Machine Learning algorithms work with numeric data and we cannot use the provided text data "as is". There are many ways to transform text data to numeric vectors. In this task you will try to use two of them.

#### Bag of words

One of the well-known approaches is a *bag-of-words* representation. To create this transformation, follow the steps:
1. Find *N* most popular words in train corpus and numerate them. Now we have a dictionary of the most popular words.
2. For each title in the corpora create a zero vector with the dimension equals to *N*.
3. For each text in the corpora iterate over words which are in the dictionary and increase by 1 the corresponding coordinate.

Let's try to do it for a toy example. Imagine that we have *N* = 4 and the list of the most popular words is 

    ['hi', 'you', 'me', 'are']

Then we need to numerate them, for example, like this: 

    {'hi': 0, 'you': 1, 'me': 2, 'are': 3}

And we have the text, which we want to transform to the vector:

    'hi how are you'

For this text we create a corresponding zero vector 

    [0, 0, 0, 0]
    
And iterate over all words, and if the word is in the dictionary, we increase the value of the corresponding position in the vector:

    'hi':  [1, 0, 0, 0]
    'how': [1, 0, 0, 0] # word 'how' is not in our dictionary
    'are': [1, 0, 0, 1]
    'you': [1, 1, 0, 1]

The resulting vector will be 

    [1, 1, 0, 1]
   
Implement the described encoding in the function *my_bag_of_words* with the size of the dictionary equals to 5000. To find the most common words use train data. You can test your code using the function *test_my_bag_of_words*.

In [ ]:
DICT_SIZE = 5000
mostFreqTupleWordsList = sorted(words_counts.items(), key=lambda x: x[1], reverse=True)[:5000]
mostFreqWordsList = [w for (w,count) in mostFreqTupleWordsList]
WORDS_TO_INDEX = dict(zip(mostFreqWordsList, range(0, len(mostFreqWordsList))))
INDEX_TO_WORDS = dict(zip(range(0, len(mostFreqWordsList)), mostFreqWordsList))

ALL_WORDS = WORDS_TO_INDEX.keys()

def my_bag_of_words(text, words_to_index, dict_size):
    """
        text: a string
        dict_size: size of the dictionary
        
        return a vector which is a bag-of-words representation of 'text'
    """
    result_vector = np.zeros(dict_size)

    # transform text string in a list of word
    wordsList = re.sub(r'\s+', " ", text).split()

    # iterate over all words in the list, and if the word is in the dictionary, we increase the value of the corresponding position in the vector
    for word in wordsList:
        if word in words_to_index.keys():
            index = words_to_index[word]
            # increment position index in the vector
            result_vector[index]+=1

    return result_vector

In [ ]:
def test_my_bag_of_words():
    words_to_index = {'hi': 0, 'you': 1, 'me': 2, 'are': 3}
    examples = ['hi how are you']
    answers = [[1, 1, 0, 1]]
    for ex, ans in zip(examples, answers):
        if (my_bag_of_words(ex, words_to_index, 4) != ans).any():
            return "Wrong answer for the case: '%s'" % ex
    return 'Basic tests are passed.'

In [ ]:
print(test_my_bag_of_words())

Now apply the implemented function to all samples (this might take up to a minute):

In [ ]:
from scipy import sparse as sp_sparse

In [ ]:
X_train_mybag = sp_sparse.vstack([sp_sparse.csr_matrix(my_bag_of_words(text, WORDS_TO_INDEX, DICT_SIZE)) for text in X_train])
X_val_mybag = sp_sparse.vstack([sp_sparse.csr_matrix(my_bag_of_words(text, WORDS_TO_INDEX, DICT_SIZE)) for text in X_val])
X_test_mybag = sp_sparse.vstack([sp_sparse.csr_matrix(my_bag_of_words(text, WORDS_TO_INDEX, DICT_SIZE)) for text in X_test])
print('X_train shape ', X_train_mybag.shape)
print('X_val shape ', X_val_mybag.shape)
print('X_test shape ', X_test_mybag.shape)

As you might notice, we transform the data to sparse representation, to store the useful information efficiently. There are many [types](https://docs.scipy.org/doc/scipy/reference/sparse.html) of such representations, however sklearn algorithms can work only with [csr](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.html#scipy.sparse.csr_matrix) matrix, so we will use this one.

**Task 3 (BagOfWords).** For the 11th row in *X_train_mybag* find how many non-zero elements it has. In this task the answer (variable *non_zero_elements_count*) should be an integer number, e.g. 20.

In [ ]:
row = X_train_mybag[10].toarray()[0]

non_zero_elements_count = np.count_nonzero(row)

print(non_zero_elements_count)

#### TF-IDF

The second approach extends the bag-of-words framework by taking into account total frequencies of words in the corpora. It helps to penalize too frequent words and provide better features space. 

Implement function *tfidf_features* using class [TfidfVectorizer](http://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) from *scikit-learn*. Use *train* corpus to train a vectorizer. Don't forget to take a look into the arguments that you can pass to it. We suggest that you filter out too rare words (occur less than in 5 titles) and too frequent words (occur more than in 90% of the titles). Also, use bigrams along with unigrams in your vocabulary. 

In [ ]:
# !pip3 install scikit-learn

from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
def tfidf_features(X_train, X_val, X_test):
    """
        X_train, X_val, X_test — samples        
        return TF-IDF vectorized representation of each sample and vocabulary
    """
    # Create TF-IDF vectorizer with a proper parameters choice
    # Fit the vectorizer on the train set
    # Transform the train, test, and val sets and return the result
    
    # Simplification: we are considering that a word apears only once in each title
    # Total number of titles: 10000
    # min_df - 5 titles in 10000 : 0.0005 (or 5 absolute value) min_df=5
    # max_df - 90% of the titles : 0.9 (or 9000 absolute value) max_df=9000
    # stop_words='english' was already removed in preprocessing; addionaly, max_df can be set to a value 
    #  in the range [0.7, 1.0) to automatically detect and filter stop words based on intra corpus document frequency of terms
    tfidf_vectorizer = TfidfVectorizer(min_df=5, max_df=9000, ngram_range=(1,2), token_pattern=r'(\S+)')
    
    X_train = tfidf_vectorizer.fit_transform(X_train)
    # print(tfidf_vectorizer.get_feature_names())
    # print(tfidf_vectorizer.vocabulary_)
    

    X_val = tfidf_vectorizer.transform(X_val)
    X_test = tfidf_vectorizer.transform(X_test)
    
    return X_train, X_val, X_test, tfidf_vectorizer.vocabulary_

In [ ]:

print(X_train[:3])

Once you have done text preprocessing, always have a look at the results. Be very careful at this step, because the performance of future models will drastically depend on it. 

In this case, check whether you have c++ or c# in your vocabulary, as they are obviously important tokens in our tags prediction task:

In [ ]:
X_train_tfidf, X_val_tfidf, X_test_tfidf, tfidf_vocab = tfidf_features(X_train, X_val, X_test)
tfidf_reversed_vocab = {i:word for word,i in tfidf_vocab.items()}

In [ ]:

print(tfidf_vocab['app'])
print(tfidf_vocab['c++'])
print(tfidf_vocab['r'])
print(tfidf_vocab['81'])
karr = np.array(list(tfidf_vocab.keys()))
print(karr[0:40])


If you can't find it, we need to understand how did it happen that we lost them? It happened during the built-in tokenization of TfidfVectorizer. Luckily, we can influence on this process. Get back to the function above and use '(\S+)' regexp as a *token_pattern* in the constructor of the vectorizer.  

In order to include words like 'c++' and 'c#' it is necessary to define the parameter token_pattern=r'(\S+)' in 
TfidfVectorizer constructor. The function tfidf_features already has this improvement.

### MultiLabel classifier

As we have noticed before, in this task each example can have multiple tags. To deal with such kind of prediction, we need to transform labels in a binary form and the prediction will be a mask of 0s and 1s. For this purpose it is convenient to use [MultiLabelBinarizer](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MultiLabelBinarizer.html) from *sklearn*.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
mlb = MultiLabelBinarizer(classes=sorted(tags_counts.keys()))
y_train = mlb.fit_transform(y_train)
y_val = mlb.fit_transform(y_val)

Implement the function *train_classifier* for training a classifier. In this task we suggest to use One-vs-Rest approach, which is implemented in [OneVsRestClassifier](http://scikit-learn.org/stable/modules/generated/sklearn.multiclass.OneVsRestClassifier.html) class. In this approach *k* classifiers (= number of tags) are trained. As a basic classifier, use [LogisticRegression](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html). It is one of the simplest methods, but often it performs good enough in text classification tasks. It might take some time, because a number of classifiers to train is large.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier

In [ ]:
def train_classifier(X_train, y_train, my_solver='lbfgs', my_penalty='l2', my_C=1.0, my_max_iter=200):
    """
      X_train, y_train — training data
      
      return: trained classifier
    """
    
    # Create and fit LogisticRegression wraped into OneVsRestClassifier.   

    # solver: optimization algorithm
    # penalty: L1 and L2-regularization
    # C: Inverse of regularization strength; must be a positive float. Like in support vector machines, smaller values specify stronger regularization
    classifier = OneVsRestClassifier(LogisticRegression(solver=my_solver,penalty=my_penalty,C=my_C,max_iter=my_max_iter)).fit(X_train, y_train)
    return classifier

Train the classifiers for different data transformations: *bag-of-words* and *tf-idf*.

If you receive a convergence warning, please set parameter *max_iter* in LogisticRegression to a larger value (the default is 100).

In [ ]:
classifier_mybag = train_classifier(X_train_mybag, y_train)
classifier_tfidf = train_classifier(X_train_tfidf, y_train)

Now you can create predictions for the data. You will need two types of predictions: labels and scores.

In [ ]:
y_val_predicted_labels_mybag = classifier_mybag.predict(X_val_mybag) # return class labels which fit > 0
y_val_predicted_scores_mybag = classifier_mybag.decision_function(X_val_mybag)

y_val_predicted_labels_tfidf = classifier_tfidf.predict(X_val_tfidf)
y_val_predicted_scores_tfidf = classifier_tfidf.decision_function(X_val_tfidf)

Now take a look at how classifier, which uses TF-IDF, works for a few examples:

In [ ]:
y_val_pred_inversed = mlb.inverse_transform(y_val_predicted_labels_tfidf)
y_val_inversed = mlb.inverse_transform(y_val)
for i in range(200,220):
    print('Title:\t{}\nTrue labels:\t{}\nPredicted labels:\t{}\n\n'.format(
        X_val[i],
        ','.join(y_val_inversed[i]),
        ','.join(y_val_pred_inversed[i])
    ))

Now, we would need to compare the results of different predictions, e.g. to see whether TF-IDF transformation helps or to try different regularization techniques in logistic regression. For all these experiments, we need to setup evaluation procedure. 

### Evaluation

To evaluate the results we will use several classification metrics:
 - [Accuracy](http://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
 - [F1-score](http://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html)
 - [Area under ROC-curve](http://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html)
 - [Area under precision-recall curve](http://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html#sklearn.metrics.average_precision_score) 
 
Make sure you are familiar with all of them. How would you expect the things work for the multi-label scenario? Read about micro/macro/weighted averaging following the sklearn links provided above.

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score 
from sklearn.metrics import average_precision_score
from sklearn.metrics import recall_score

Implement the function *print_evaluation_scores* which calculates and prints to stdout:
 - *accuracy*
 - *F1-score macro/micro/weighted*
 - *Precision macro/micro/weighted*

In [ ]:
def print_evaluation_scores(title, y_val, predicted):

    print('### ', title)
    print('Accuracy: ', accuracy_score(y_val, predicted))
    print('F1-score micro: ', f1_score(y_val, predicted, average='micro'))
    print('F1-score macro: ', f1_score(y_val, predicted, average='macro'))
    print('F1-score weighted: ', f1_score(y_val, predicted, average='weighted'))
    print('Average precision score micro: ', average_precision_score(y_val, predicted, average='micro'))
    print('Average precision score macro: ', average_precision_score(y_val, predicted, average='macro'))
    print('Average precision score weighted: ', average_precision_score(y_val, predicted, average='weighted'))


In [ ]:

print_evaluation_scores('Bag-of-words', y_val, y_val_predicted_labels_mybag)
print_evaluation_scores('Tfidf', y_val, y_val_predicted_labels_tfidf)

You might also want to plot some generalization of the [ROC curve](http://scikit-learn.org/stable/modules/model_evaluation.html#receiver-operating-characteristic-roc) for the case of multi-label classification. Provided function *roc_auc* can make it for you. The input parameters of this function are:
 - true labels
 - decision functions scores
 - number of classes

In [ ]:

from metrics import roc_auc
from sklearn.metrics import roc_auc_score
%matplotlib inline

In [ ]:
n_classes = len(tags_counts)
roc_auc(y_val, y_val_predicted_scores_mybag, n_classes)
roc_auc_score(y_val, y_val_predicted_scores_mybag)

In [ ]:
n_classes = len(tags_counts)
roc_auc(y_val, y_val_predicted_scores_tfidf, n_classes)
roc_auc_score(y_val, y_val_predicted_scores_tfidf)

**Task 4 (MultilabelClassification).** Once we have the evaluation set up, we suggest that you experiment a bit with training your classifiers. We will use *F1-score weighted* as an evaluation metric. Recommendation:
- compare the quality of the bag-of-words and TF-IDF approaches and chose one of them.
- for the chosen one, try *L1* and *L2*-regularization techniques in Logistic Regression with different coefficients (e.g. C equal to 0.1, 1, 10, 100).

You also could try other improvements of the preprocessing / model, if you want. 

In [ ]:


from contextlib import redirect_stdout

"""
listOfparams = [
    {'solver': 'liblinear', 'penalty': 'l1', 'C': 0.1},
    {'solver': 'liblinear', 'penalty': 'l1', 'C': 1.0},
    {'solver': 'liblinear', 'penalty': 'l1', 'C': 10.0},
    {'solver': 'liblinear', 'penalty': 'l1', 'C': 100.0},
    {'solver': 'saga', 'penalty': 'l1', 'C': 0.1},
    {'solver': 'saga', 'penalty': 'l1', 'C': 1.0},
    {'solver': 'saga', 'penalty': 'l1', 'C': 10.0},
    {'solver': 'saga', 'penalty': 'l1', 'C': 100.0},
    {'solver': 'liblinear', 'penalty': 'l2', 'C': 0.1},
    {'solver': 'liblinear', 'penalty': 'l2', 'C': 1.0},
    {'solver': 'liblinear', 'penalty': 'l2', 'C': 10.0},
    {'solver': 'liblinear', 'penalty': 'l2', 'C': 100.0},
    {'solver': 'newton-cg', 'penalty': 'l2', 'C': 0.1},
    {'solver': 'newton-cg', 'penalty': 'l2', 'C': 1.0},
    {'solver': 'newton-cg', 'penalty': 'l2', 'C': 10.0},
    {'solver': 'newton-cg', 'penalty': 'l2', 'C': 100.0},
    {'solver': 'lbfgs', 'penalty': 'l2', 'C': 0.1},
    {'solver': 'lbfgs', 'penalty': 'l2', 'C': 1.0},
    {'solver': 'lbfgs', 'penalty': 'l2', 'C': 10.0},
    {'solver': 'lbfgs', 'penalty': 'l2', 'C': 100.0},
    {'solver': 'sag', 'penalty': 'l2', 'C': 0.1},
    {'solver': 'sag', 'penalty': 'l2', 'C': 1.0},
    {'solver': 'sag', 'penalty': 'l2', 'C': 10.0},
    {'solver': 'sag', 'penalty': 'l2', 'C': 100.0},
    {'solver': 'saga', 'penalty': 'l2', 'C': 0.1},
    {'solver': 'saga', 'penalty': 'l2', 'C': 1.0},
    {'solver': 'saga', 'penalty': 'l2', 'C': 10.0},
    {'solver': 'saga', 'penalty': 'l2', 'C': 100.0}
]
"""

listOfparams = [
    {'solver': 'lbfgs', 'penalty': 'l2', 'C': 10.0, 'my_max_iter':350}
]

with open('results.md', 'a') as f:
    with redirect_stdout(f):

        for params in listOfparams:
            print('## ', str(params).replace("{","\{").replace("}","\}"))
            # train the classifieres with different data transformations: bag-of-words and tf-idf
            classifier_mybag = train_classifier(X_train_mybag, y_train, params['solver'], params['penalty'], params['C'], params['my_max_iter'])
            classifier_tfidf = train_classifier(X_train_tfidf, y_train, params['solver'], params['penalty'], params['C'], params['my_max_iter'])

            # create predictions for the data. You will need two types of predictions: labels and scores.
            y_val_predicted_labels_mybag = classifier_mybag.predict(X_val_mybag)
            y_val_predicted_scores_mybag = classifier_mybag.decision_function(X_val_mybag)

            y_val_predicted_labels_tfidf = classifier_tfidf.predict(X_val_tfidf)
            y_val_predicted_scores_tfidf = classifier_tfidf.decision_function(X_val_tfidf)

            # evaluate each of the classifiers
            print_evaluation_scores('Bag-of-words' + ' ' + str(params).replace("{","\{").replace("}","\}"), y_val, y_val_predicted_labels_mybag)
            print_evaluation_scores('Tfidf' + ' ' + str(params).replace("{","\{").replace("}","\}"), y_val, y_val_predicted_labels_tfidf)

print('Finished.')

In [ ]:

test_predictions = classifier_tfidf.predict(X_test_tfidf)
test_pred_inversed = mlb.inverse_transform(test_predictions)

test_predictions_for_print = '\n'.join('%i\t%s' % (i, ','.join(row)) for i, row in enumerate(test_pred_inversed))

print(test_predictions_for_print)

### Analysis of the most important features

Finally, it is usually a good idea to look at the features (words or n-grams) that are used with the largest weigths in your logistic regression model.

Implement the function *print_words_for_tag* to find them. Get back to sklearn documentation on [OneVsRestClassifier](http://scikit-learn.org/stable/modules/generated/sklearn.multiclass.OneVsRestClassifier.html) and [LogisticRegression](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) if needed.

In [ ]:
def print_words_for_tag(classifier, tag, tags_classes, index_to_words, all_words):
    """
        classifier: trained classifier
        tag: particular tag
        tags_classes: a list of classes names from MultiLabelBinarizer
        index_to_words: index_to_words transformation
        all_words: all words in the dictionary
        
        return nothing, just print top 5 positive and top 5 negative words for current tag
    """
    print('Tag:\t{}'.format(tag))
    
    # Extract an estimator from the classifier for the given tag.
    # Extract from the OneVsRestClassifier the Logistic Regression estimator corresponding to the tag
    estimator = classifier.estimators_[mlb.classes.index(tag)]
    # print(estimator.coef_)
    # print(estimator.coef_.shape)
    # print(len(index_to_words))
    # print(type(estimator.coef_))

    # Extract feature coefficients from the estimator. 
    coef = estimator.coef_.flatten()
    print('coef=', coef)
    # Get indices of the top-5 coeficients 
    ind_top5 = np.argpartition(coef, -5)[-5:]
    ind_top5 = ind_top5[np.argsort(coef[ind_top5])]
    # top-5 coeficients: coef[ind_top5]
    print('top-5 coeficients:', coef[ind_top5])
    print('top-5 coeficients indices:', ind_top5)
    # Get indices of the bottom-5 coeficients
    ind_bottom5 = np.argpartition(coef, 5)[:5]
    ind_bottom5 = ind_bottom5[np.argsort(coef[ind_bottom5])]
    # bottom-5 coeficients: coef[ind_bottom5]
    print('bottom-5 coeficients:', coef[ind_bottom5])
    print('bottom-5 coeficients indices:', ind_bottom5)

    #top_positive_words = # top-5 words sorted by the coefficiens.
    top_positive_words = [ index_to_words[ind] for ind in ind_top5[::-1]]
    #top_negative_words = # bottom-5 words  sorted by the coefficients.
    top_negative_words = [ index_to_words[ind] for ind in ind_bottom5]
    print('Top positive words:\t{}'.format(', '.join(top_positive_words)))
    print('Top negative words:\t{}\n'.format(', '.join(top_negative_words)))

In [ ]:
print_words_for_tag(classifier_tfidf, 'c', mlb.classes, tfidf_reversed_vocab, ALL_WORDS)
print_words_for_tag(classifier_tfidf, 'c++', mlb.classes, tfidf_reversed_vocab, ALL_WORDS)
print_words_for_tag(classifier_tfidf, 'linux', mlb.classes, tfidf_reversed_vocab, ALL_WORDS)